In [ ]:
"""Cloud inference entrypoint for a locally trained BigAlpha Transformer."""

from __future__ import annotations

import json
import os


def main(datasources, start_date, end_date):
    import numpy as np
    import pandas as pd
    import torch
    from torch import nn
    import dai

    model_candidates = [os.path.join(os.getcwd(), "transformer_model.json")]
    try:
        import transformer_train

        model_candidates.append(
            os.path.join(
                os.path.dirname(os.path.abspath(transformer_train.__file__)),
                "transformer_model.json",
            )
        )
    except ImportError:
        pass
    MODEL_PATH = next(
        (path for path in model_candidates if os.path.exists(path)),
        None,
    )
    if MODEL_PATH is None:
        raise FileNotFoundError(
            f"missing model checkpoint; searched={model_candidates}"
        )

    PRICE_COLS = ["open", "high", "low", "close", "bid_price1", "ask_price1"]
    VOLUME_COLS = ["volume", "amount", "bid_volume1", "ask_volume1"]
    FEATURE_COLS = PRICE_COLS + VOLUME_COLS

    def load_checkpoint(path):
        with open(path, "r", encoding="utf-8") as handle:
            payload = json.load(handle)
        state = {}
        for name, item in payload["state_dict"].items():
            dtype = getattr(torch, item["dtype"])
            state[name] = torch.tensor(item["data"], dtype=dtype).reshape(item["shape"])
        payload["state_dict"] = state
        return payload

    checkpoint = load_checkpoint(MODEL_PATH)
    config = checkpoint["model_cfg"]
    if checkpoint["feature_cols"] != FEATURE_COLS:
        raise RuntimeError("checkpoint feature columns do not match inference code")

    class StockTransformer(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            self.pooling = cfg.get("pooling", "mean")
            if self.pooling not in {"mean", "last", "attention"}:
                raise ValueError(f"unsupported pooling mode: {self.pooling}")
            self.proj = nn.Linear(cfg["n_feat"], cfg["d_model"])
            self.pos = nn.Parameter(torch.zeros(1, cfg["seq_len"], cfg["d_model"]))
            layer = nn.TransformerEncoderLayer(
                d_model=cfg["d_model"], nhead=cfg["nhead"], dim_feedforward=cfg["dim_ff"],
                dropout=cfg["dropout"], batch_first=True, norm_first=True, activation="gelu"
            )
            self.encoder = nn.TransformerEncoder(layer, cfg["nlayers"], norm=nn.LayerNorm(cfg["d_model"]))
            if self.pooling == "attention":
                self.pool_query = nn.Parameter(torch.zeros(cfg["d_model"]))
            self.head = nn.Sequential(nn.LayerNorm(cfg["d_model"]), nn.Linear(cfg["d_model"], 1))

        def forward(self, x):
            hidden = self.encoder(self.proj(x) + self.pos)
            if self.pooling == "last":
                pooled = hidden[:, -1]
            elif self.pooling == "attention":
                logits = torch.einsum("bsd,d->bs", hidden, self.pool_query)
                logits = logits / np.sqrt(hidden.shape[-1])
                weights = torch.softmax(logits, dim=1)
                pooled = torch.einsum("bs,bsd->bd", weights, hidden)
            else:
                pooled = hidden.mean(dim=1)
            return self.head(pooled).squeeze(-1)

    datasource_key = checkpoint.get("datasource_key")
    if not datasource_key:
        raise RuntimeError("checkpoint is missing datasource_key")
    if datasource_key not in datasources:
        raise KeyError(f"missing datasource {datasource_key!r}; available={list(datasources)}")
    table = datasources[datasource_key]
    buffer_start = (pd.Timestamp(start_date) - pd.Timedelta(days=20)).strftime("%Y-%m-%d")
    sql = f"SELECT date, instrument, {', '.join(FEATURE_COLS)} FROM {table} ORDER BY instrument, date"
    frame = dai.query(sql, filters={"date": [buffer_start, str(end_date)]}, compression=True).df()
    frame["date"] = pd.to_datetime(frame["date"])
    frame["key"] = frame["instrument"].astype(str)
    for column in VOLUME_COLS:
        frame[column] = np.log1p(frame[column].clip(lower=0))

    seq_len = int(config["seq_len"])
    start_ts, end_ts = pd.Timestamp(start_date).normalize(), pd.Timestamp(end_date).normalize()
    windows, keys = [], []
    for key, sub in frame.groupby("key", sort=False, observed=True):
        sub = sub.sort_values("date", kind="mergesort").reset_index(drop=True)
        values = sub[FEATURE_COLS].to_numpy(dtype=np.float32)
        day = sub["date"].dt.normalize().to_numpy()
        endpoints = np.flatnonzero(np.r_[day[1:] != day[:-1], True])
        for endpoint in endpoints:
            current_day = pd.Timestamp(day[endpoint])
            window_start = endpoint - seq_len + 1
            if current_day < start_ts or current_day > end_ts or window_start < 0:
                continue
            windows.append(values[window_start : endpoint + 1])
            keys.append((current_day, str(key)))
    if not windows:
        raise RuntimeError("no inference samples built")

    x = np.stack(windows).astype(np.float32)
    mean = np.asarray(checkpoint["mean"], dtype=np.float32)
    scale = np.asarray(checkpoint["std"], dtype=np.float32)
    x = np.where(np.isfinite(x), x, mean.reshape(1, 1, -1))
    x = ((x - mean.reshape(1, 1, -1)) / scale.reshape(1, 1, -1)).astype(np.float32)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = StockTransformer(config).to(device)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    output = []
    tensor = torch.from_numpy(x)
    batch_size = 1024
    with torch.no_grad():
        for offset in range(0, len(tensor), batch_size):
            batch = tensor[offset : offset + batch_size].to(device)
            with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
                output.append(model(batch).float().cpu().numpy())

    result = pd.DataFrame(keys, columns=["date", "instrument"])
    result["score"] = np.concatenate(output).astype(np.float64)
    universe = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]}, compression=True
    ).df()
    universe["date"] = pd.to_datetime(universe["date"]).dt.normalize()
    result = (
        result.merge(universe, on=["date", "instrument"], how="inner")
        .replace([np.inf, -np.inf], np.nan).dropna(subset=["score"])
        .drop_duplicates(["date", "instrument"])[["date", "instrument", "score"]]
        .sort_values(["date", "instrument"], kind="mergesort").reset_index(drop=True)
    )
    if result.empty:
        raise RuntimeError("empty score result")
    return result
